---

# 🎁 도전과제

## 도전과제 1. 페르소나 + 메모리 챗봇

페르소나 챗봇에 **메모리**를 결합하세요.  
"내 이름은 X" → "내 이름이 뭐였지?"가 페르소나 톤으로 자연스럽게 이어져야 합니다.

**힌트**:
- 프롬프트 system 메시지에 `{persona}` 변수 자리 만들기
- `MessagesPlaceholder("history")` 끼우기
- `RunnableWithMessageHistory`로 감싸기


In [1]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab 환경입니다.")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter 환경입니다.")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 로드 완료" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")

from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, openai_api_key=os.environ.get("OPENAI_API_KEY"))
print("✅ LLM 준비 완료")

✅ 로컬 Jupyter 환경입니다.
✅ API 키 로드 완료
✅ LLM 준비 완료


In [2]:
# 📌 페르소나 + 메모리 챗봇 — 빈칸을 채우세요
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 👇 빈칸 1: 프롬프트 — system에 {persona}, history 자리, human에 {question}
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다. 이 페르소나의 말투와 태도를 일관되게 유지하세요."),
    MessagesPlaceholder(variable_name="history"),   # ← 대화 기록이 들어갈 자리
    ("human", "{question}"),
])

# 세션 저장소 (재사용)
persona_store = {}
def get_persona_history(session_id):
    if session_id not in persona_store:
        persona_store[session_id] = InMemoryChatMessageHistory()
    return persona_store[session_id]

# 👇 빈칸 2: 체인을 메모리로 감싸기
persona_chain = RunnableWithMessageHistory(
    persona_prompt | llm,
    get_persona_history,
    input_messages_key="question",   # ← 매번 새로 들어오는 사용자 입력 키
    history_messages_key="history",  # ← 위 MessagesPlaceholder와 같은 이름!
)


def chat_with_persona(session_id: str, persona: str, question: str) -> str:
    """페르소나·세션·질문을 받아 답변 반환"""
    result = persona_chain.invoke(
        {"persona": persona, "question": question},
        config={"configurable": {"session_id": session_id}},
    )
    return result.content


# 테스트 — 같은 세션에서 3턴 대화
sid = "test_user"
persona = "냉정하지만 친절한 시니어 데이터 분석가"
print("[1턴]", chat_with_persona(sid, persona, "안녕하세요. 저는 진환이라고 합니다."))
print("[2턴]", chat_with_persona(sid, persona, "표본 분산과 모분산의 차이를 한 줄로 알려주세요."))
print("[3턴]", chat_with_persona(sid, persona, "제가 누구라고 했었죠?"))


c:\Users\user\anaconda3\envs\day01_venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


[1턴] 안녕하세요, 진환님. 만나서 반갑습니다. 데이터 분석과 관련하여 궁금한 점이나 도움이 필요하신 부분이 있으신가요?
[2턴] 표본 분산은 표본 데이터의 분산을 나타내고, 모분산은 전체 모집단의 분산을 나타내며, 표본 분산은 모집단 분산을 추정하기 위해 표본 크기로 나누는 방식이 다릅니다.
[3턴] 진환님이라고 하셨습니다. 추가로 궁금한 점이나 논의하고 싶은 주제가 있으신가요?


In [3]:
persona_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {persona}입니다. 이 페르소나의 말투와 태도를 일관되게 유지하세요."),
    MessagesPlaceholder(variable_name="history"),   # ← 대화 기록이 들어갈 자리
    ("human", "{question}"),
])

persona_chain = RunnableWithMessageHistory(
    persona_prompt | llm,
    get_persona_history,
    input_messages_key="question",   # ← 매번 새로 들어오는 사용자 입력 키
    history_messages_key="history",  # ← 위 MessagesPlaceholder와 같은 이름!
)

c:\Users\user\anaconda3\envs\day01_venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [ ]:
def run_chatbot(initial_persona: str = "분석가"):
    persona = initial_persona
    session_id = "chat_user"

    print(f"👋 페르소나 챗봇입니다. (현재: {persona})")
    print("종료하려면 /quit을 입력하세요.")

    while True:
        user_input = input("질문> ").strip()

        if not user_input:
            continue

        if user_input == "/quit":
            break

        answer = chat_with_persona(
            session_id=session_id,
            persona=persona,
            question=user_input,
        )
        print(f"[{persona}] {answer}\n")

In [9]:
run_chatbot()

👋 페르소나 챗봇입니다. (현재: 분석가)
종료하려면 /quit을 입력하세요.
[분석가] 안녕하세요! 어떻게 도와드릴까요?

[분석가] 반갑습니다, 재연님! 어떤 이야기를 나누고 싶으신가요?

[분석가] 재연님이라고 하셨죠! 맞나요?

[분석가] 감사합니다! 재연님, 어떤 주제로 이야기해볼까요?



✅ **결과 확인**: 3턴 답변에 "진환"이 나오면서 페르소나 톤이 유지되어야 합니다.